In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()


In [ ]:
# Question 70: Find employees whose salary has decreased (Self Join).
# Concept: Detecting anomalies
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
salary_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.salary",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
# Solution:
s1 = salary_df.alias("s1")
s2 = salary_df.alias("s2")

s1.join(s2, (col("s1.employee_id") == col("s2.employee_id")) & 
           (col("s2.from_date") > col("s1.from_date")) & 
           (col("s2.amount") < col("s1.amount"))) \
   .select(col("s1.employee_id")) \
   .distinct() \
   .show()
